# Task 3: Research Results, Data Analysis, and Scientific Visualization
**Project:** Fast Dopamine & Academic Performance — GPA Prediction Benchmark  
**Repository:** `https://github.com/EraMiir/fast-dopamine-academic-performance`

This notebook executes the complete reproducible analysis pipeline:
1. **Data Ingestion & Warm-up Filtering:** Loading raw benchmark logs (`data/raw/test_run.jsonl`) and discarding cold-start `warmup` runs.
2. **Statistical Aggregation:** Computing Mean, SD, Median, Percentiles ($p_{25}, p_{75}, p_{95}, p_{99}$), Bootstrapped 95% Confidence Intervals (CI), and Guardrail Error Rates.
3. **Scientific Visualization & Export:** Generating colorblind-safe plots with explicit physical units, log scales, and 95% CI bands exported to `.pdf` and 300 DPI `.png`.

In [1]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)

raw_path = '../data/raw/test_run.jsonl'
raw_df = pd.read_json(raw_path, lines=True)
eval_df = raw_df[raw_df['phase'] == 'evaluation'].copy()

print(f'Total raw records loaded: {len(raw_df)}')
print(f'Warm-up records discarded: {len(raw_df) - len(eval_df)}')
print(f'Clean evaluation records retained: {len(eval_df)}')

FileNotFoundError: File ../data/raw/test_run.jsonl does not exist

In [ ]:
def bootstrap_ci(values, n_boot=1000, seed=42):
    rng = np.random.default_rng(seed)
    boot_means = [np.mean(rng.choice(values, size=len(values), replace=True)) for _ in range(n_boot)]
    return float(np.percentile(boot_means, 2.5)), float(np.percentile(boot_means, 97.5))

summary_rows = []
for (model, sample_size), grp in eval_df.groupby(['model', 'sample_size']):
    rmse_ci_l, rmse_ci_h = bootstrap_ci(grp['RMSE'].values)
    lat_ci_l, lat_ci_h = bootstrap_ci(grp['latency_ms'].values)
    summary_rows.append({
        'model': model,
        'sample_size': int(sample_size),
        'n_runs': len(grp),
        'RMSE_mean': round(float(grp['RMSE'].mean()), 4),
        'RMSE_sd': round(float(grp['RMSE'].std()), 4),
        'RMSE_median': round(float(grp['RMSE'].median()), 4),
        'RMSE_p25': round(float(np.percentile(grp['RMSE'], 25)), 4),
        'RMSE_p75': round(float(np.percentile(grp['RMSE'], 75)), 4),
        'RMSE_p95': round(float(np.percentile(grp['RMSE'], 95)), 4),
        'RMSE_p99': round(float(np.percentile(grp['RMSE'], 99)), 4),
        'RMSE_ci95_low': round(rmse_ci_l, 4),
        'RMSE_ci95_high': round(rmse_ci_h, 4),
        'MAE_mean': round(float(grp['MAE'].mean()), 4),
        'MAE_sd': round(float(grp['MAE'].std()), 4),
        'latency_mean_ms': round(float(grp['latency_ms'].mean()), 3),
        'latency_sd_ms': round(float(grp['latency_ms'].std()), 3),
        'latency_median_ms': round(float(grp['latency_ms'].median()), 3),
        'latency_p95_ms': round(float(np.percentile(grp['latency_ms'], 95)), 3),
        'latency_p99_ms': round(float(np.percentile(grp['latency_ms'], 99)), 3),
        'latency_ci95_low_ms': round(lat_ci_l, 3),
        'latency_ci95_high_ms': round(lat_ci_h, 3),
        'ram_mean_mb': round(float(grp['ram_mb'].mean()), 2),
        'error_rate': round(float(grp['error_rate'].mean()), 4)
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv('../data/processed/metrics_summary.csv', index=False)
print('Aggregated Metrics Summary (Sample Size N=256):')
print(summary_df[summary_df['sample_size'] == 256][['model', 'RMSE_mean', 'RMSE_sd', 'RMSE_median', 'RMSE_p95', 'latency_mean_ms', 'latency_p95_ms', 'error_rate']].to_string(index=False))

In [ ]:
from pathlib import Path
out_dir = Path('../reports/figures')
out_dir.mkdir(parents=True, exist_ok=True)

import sys
sys.path.append('../src')
from visualize import generate_all_plots
generate_all_plots(eval_df, summary_df, str(out_dir))
print('Successfully exported fig1_latency_vs_load (.pdf & .png) and fig2_distribution_boxplot (.pdf & .png) to ../reports/figures/')